# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** Daniel Eta
**Student ID:** 43092028

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [3]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
from dotenv import load_dotenv
load_dotenv()
API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
# from google.colab import userdata
# API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [7]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response
#
# TODO: Call it once with a simple question and print the answer.
response = ask_llm("What can you do?")
print(response.choices[0].message.content)

# TODO: Print response.usage as well — how many tokens did your call consume?
print("\n")
print(response.usage.total_tokens, " tokens were used.")

I can be used in a variety of ways, from helping you plan a vacation to creating art. I'm here to assist you in finding the help or information you need. My strengths include answering questions, generating text and images and even just chatting with you.


99  tokens were used.


**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:**
> 1. System roles help provide the high-level guard-rails with which the model can operate. It gives context with regard to the role, personality, or tone of instruction the responses should be tailored to e.g. "You are a calm tech support assistant. Your job is to fix software bugs with clear steps." On the other hand, user roles (i.e. "generate a python script that scrapes the following websites") provide the specific task that the agent should respond to. Ideally, they work together. System role to specify how forthcoming responses should be tailored; user roles to request what specific responses need to be provided.
> 2. A token is the fundamental unit read or generated by a language model. It is rationale to adopt token-based pricing because it allows an atomic, standard rate of pricing amongst all users independent of user behaviour or patterns. If pricing was request-based, users that sent very long prompt asking for convoluted answers would be at a much higher advantage than users who sent simple prompts frequently. The flat-fee aligns token use costs with the actual hardware costs in running the model.

### Part 1.2 — Temperature: the randomness dial

In [8]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
# TODO: Print all 10 answers, grouped by temperature.
prompt = "Suggest a name for a savings product for market traders in Accra."
temperatures = [0.0, 1.2]

for temp in temperatures:
    print("Question at t = " + str(temp))
    for i in range(1, 6):
        response = ask_llm(prompt, temperature=temp)
        print(f"Run {i}:")
        print(response.choices[0].message.content)
        print("")
    print()

Question at t = 0.0
Run 1:
Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Traders' Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Accra Amanfu**: "Amanfu" is a Ghanaian word for "savings" or "treasury", so this name incorporates local language and culture.
4. **Market Mobi**: This name is short and catchy, and "Mobi" implies mobility and flexibility, which could appeal to market traders who need to manage their finances on-the-go.
5. **Sika Su**: "Sika" is the Ghanaian word for "money", and "Su" means "grow" or "increase", so this name suggests a savings product that helps traders grow their wealth.
6. **Kokroko Savings**: "Kokroko" is a Ghanaian word for "honest" or "trustworthy", which could convey a sense of reliability and security for market traders.
7. **Traders' Fund**: This name is straightforward 

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:** When the temperature was set to $0.0$, the text across all five runs felt repetitive. It followed a distinct format, and there was little variation. When we upped the temperature to $1.2$, the responses I got became more varied. The vocabulary was more diverse, as was the sentence format and length. For a support system for loans, I'd lean towards a lower temperature because consistency is very important for financial services. Processing loans is very formulaic behaviour, and I would not want a high temperature to cause the model to work in diverse (perhaps unpredictable) ways.

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [15]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [16]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.
SUMMARY_PROMPT_V1 = "Summarize the following loan application"

for letter_id in ["L002", "L006"]:
    user_prompt = SUMMARY_PROMPT_V1 + f"\n{LETTERS[letter_id]}"
    response_v1 = ask_llm(
        user_prompt=user_prompt,
        system_prompt="You are a helpful assistant.",
        temperature=0.0,
        max_tokens=300
    )
    print(f"({letter_id}) V1 Summary:")
    print(response_v1.choices[0].message.content)
    print("")

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.
SUMMARY_PROMPT_V2 = "You are an assistant to a microfinance loan officer. Summarise the provided loan application in 3-4 concise, neutral, and strictly factual sentences. Include the requested loan amount, stated purpose, repayment plan, and collateral status. Do not invent any details."

for letter_id in ["L002", "L006"]:
    user_prompt = "Summarise the following loan application" + f"\n{LETTERS[letter_id]}"
    response_v2 = ask_llm(
        user_prompt=user_prompt,
        system_prompt=SUMMARY_PROMPT_V2,
        temperature=0.0,
        max_tokens=300
    )
    print(f"({letter_id}) V2 Summary:")
    print(response_v2.choices[0].message.content)
    print("")
    
# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.

(L002) V1 Summary:
Here is a summary of the loan application:

* Applicant: Kwame Boateng, a commercial driver in Kumasi
* Loan amount: GHS 25,000
* Purpose: To repair his trotro engine and settle personal debts
* Repayment plan: No specific plan, but promises to pay back when he can
* Collateral: None
* Urgency: Needs the loan urgently, citing slow business that is expected to pick up after the festive season.

(L006) V1 Summary:
Here is a summary of Kofi's loan application:

* Loan amount: GHS 50,000
* Business ideas: 
  1. Car washing business
  2. Provision shop
  3. Importing phones from Dubai
* Applicant's details: 22 years old, no prior business experience, but claims to be "business-minded"
* Repayment plan: Pay back the loan in 1 year, once the businesses are established and profitable
* Collateral: None, but Kofi claims to be "trustworthy"

(L002) V2 Summary:
Kwame Boateng has applied for a loan of GHS 25,000 to repair his trotro engine and settle personal debts. The stated p

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:** The first prompt, V1, was generally inconsistent with its output. It adopted a bullet-point format, but the lengths of these points were inconsistent and some fields of interest were missing entirely. V2 helped acutely shape out the boilerplate. It specified the length and the target dimensions of what the output was meant to look like (Amount, Purpose, Repayment Plan, and Collateral). No invented details particularly important because LLMs can sometimes "hallucinate" when text prediction based on statistical patterns cause models to generate false information confidently. Intentionally specifying that text should not be invented adds another gate to guard against this.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [18]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0
import json
import pandas as pd

EXTRACT_SYSTEM_PROMPT = """You are a strict data extraction system for a microfinance institution.
Your task is to extract loan application details from the provided letter and output ONLY a valid JSON object matching this exact schema:

{
  "applicant_name": "string (full name)",
  "amount_ghs": "number (numeric value only, no currency symbols or commas)",
  "purpose": "string (brief summary of what the loan is for)",
  "monthly_profit_ghs": "number or null (current monthly profit/income, numeric only)",
  "has_collateral_or_guarantor": "boolean (true if applicant explicitly offers collateral or a guarantor, false otherwise)",
  "repayment_months": "number or null (proposed loan duration in months as an integer)"
}

Rules:
1. Output ONLY the JSON object. Do not include introductory text, explanations, or conclusions.
2. If a field is not explicitly stated in the letter, use null (or false for has_collateral_or_guarantor if none offered). Do not guess or infer missing financial figures.
3. Convert all time periods to months (e.g., 1 year = 12).

Example Application Letter:
"My name is Roronoa Zoro and I run a sword sharpening stall in Shimotsuki Market with Dracule Mihawk. I am applying for a loan of GHS 12,000 to purchase three Meito-grade katanas and expand into custom blade forging. My stall currently generates about GHS 1,500 profit each month. I plan to repay the full loan over 18 months. My training partner, Perona, has agreed to stand as my guarantor."

Example JSON Output:
{
  "applicant_name": "Roronoa Zoro",
  "amount_ghs": 12000,
  "purpose": "Purchase three Meito-grade katanas and expand blade forging workshop",
  "monthly_profit_ghs": 1500,
  "has_collateral_or_guarantor": true,
  "repayment_months": 18
}
"""

# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).
def extract_fields(letter_text):
    # Call your ask_llm helper at temperature=0
    response = ask_llm(
        user_prompt=f"Extract the fields from this loan application:\n\n{letter_text}",
        system_prompt=EXTRACT_SYSTEM_PROMPT,
        temperature=0.0,
    )

    # Get the raw text content
    raw_text = response.choices[0].message.content

    # Clean out any markdown formatting fences (```json or ```)
    cleaned_text = raw_text.replace("```json", "").replace("```", "").strip()

    # Try to parse into a Python dictionary
    try:
        return json.loads(cleaned_text)
    except json.JSONDecodeError:
        print("Warning: Could not parse response as JSON.")
        print(cleaned_text)
        return None

# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.
rows = []

for letter_id, letter_text in LETTERS.items():
    record = extract_fields(letter_text)
    if record is not None:
        record["letter_id"] = letter_id
        rows.append(record)

df = pd.DataFrame(rows)
df

,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months,letter_id
0,Akosua Mensah,8000,Buy a deep freezer and expand into frozen foods,900.0,True,20.0,L001
1,Kwame Boateng,25000,Repair trotro engine and settle personal debts,NaN,False,NaN,L002
2,Efua Darko,15000,Purchase industrial sewing machines and fabric...,2800.0,True,15.0,L003
3,Yaw Owusu,12000,Purchase feed and 500 new layers for poultry farm,1500.0,True,18.0,L004
4,Adenta Women's Weaving Cooperative,30000,Buy a bulk order of yarn to cut out middlemen ...,NaN,True,16.0,L005
5,Kofi,50000,"Start a car washing business, a provision shop...",NaN,False,12.0,L006


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:**
> Test sets are meant to stay unseen until evaluation. If we had used one letter from our bank, the model would just recall the example (overfitting) instead of generalising to the output pattern. Forcing the system to use null stops hallucination. If the model felt as though it had to provide something for each field, it could hallucinate. Giving it an option not to increases reliability. `t=0` is optimal because extraction requires the model to be very precise, formulaic, and not varied. It is not a creative task, so we just let it focus on the very systematic task.

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [ ]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.

# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:** [Double-click to edit]

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** [paste here]

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [ ]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.

### Part 4.2 — Reliability: is the system consistent?

In [ ]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

### Part 4.3 — Hallucination probing

In [ ]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

# TODO: Record the outputs verbatim below and label each PASS or FAIL.

**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:** [Double-click to edit]

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:** [Double-click to edit]

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:** [Double-click to edit]

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.